In [ ]:
import os
import random
from pathlib import Path
from IPython.display import Audio
from tqdm.auto import tqdm
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from huggingface_hub import login as hf_login
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset

load_dotenv()

ROOT_DIR = Path.home() / "workspace"
DATA_DIR = ROOT_DIR / "dataset" / "STARSS22-23"

HF_TOKEN = os.getenv("HF_TOKEN")

hf_login(token=HF_TOKEN)

In [ ]:
files = list(DATA_DIR.iterdir())
files

In [ ]:
# ============================================================
# STARSS22-23 — Full EDA
# ============================================================

# --- 1. Dataset Overview ---
print("=" * 60)
print("📁 STARSS22-23 Dataset Structure")
print("=" * 60)

METADATA_DIR = DATA_DIR / "metadata_dev"
AUDIO_DIR = DATA_DIR / "stereo_dev"
VIDEO_DIR = DATA_DIR / "video_dev"

splits = ["dev-train-sony", "dev-train-tau", "dev-test-sony", "dev-test-tau"]

# Count files per split
split_info = {}
for split in splits:
    meta_files = sorted((METADATA_DIR / split).glob("*.csv"))
    audio_files = sorted((AUDIO_DIR / split).glob("*.wav"))
    split_info[split] = {
        "metadata_files": len(meta_files),
        "audio_files": len(audio_files),
    }
    print(f"  {split}: {len(meta_files)} metadata, {len(audio_files)} audio")

# Total
total_meta = sum(v["metadata_files"] for v in split_info.values())
total_audio = sum(v["audio_files"] for v in split_info.values())
print(f"\n  Total: {total_meta} metadata, {total_audio} audio files")


# --- 2. Class Definition ---
CLASS_MAP = {
    0: "Female speech",
    1: "Male speech",
    2: "Clapping",
    3: "Telephone",
    4: "Laughter",
    5: "Domestic sounds",
    6: "Walk / Footsteps",
    7: "Door (open/close)",
    8: "Music",
    9: "Musical instrument",
    10: "Water tap",
    11: "Bell",
    12: "Knock",
}

print(f"\n🏷️ {len(CLASS_MAP)} Sound Event Classes:")
for k, v in CLASS_MAP.items():
    print(f"  {k:2d}: {v}")


# --- 3. Load ALL metadata into one DataFrame ---
print("\n" + "=" * 60)
print("📊 Loading metadata...")
print("=" * 60)

all_metadata = []
for split in splits:
    csv_files = sorted((METADATA_DIR / split).glob("*.csv"))
    for csv_file in tqdm(csv_files, desc=split):
        df = pd.read_csv(csv_file)
        df["file_name"] = csv_file.stem
        df["split"] = split
        df["source_type"] = "sony" if "sony" in split else "tau"
        df["train_test"] = "train" if "train" in split else "test"
        all_metadata.append(df)

meta_df = pd.concat(all_metadata, ignore_index=True)
meta_df["class_name"] = meta_df["class"].map(CLASS_MAP)

print(f"\nTotal annotation rows: {len(meta_df):,}")
print(f"Unique files: {meta_df['file_name'].nunique():,}")
print(f"Columns: {list(meta_df.columns)}")
print(meta_df.head(10))


# --- 4. Class Distribution ---
print("\n" + "=" * 60)
print("📊 Class Distribution (by frame count)")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Overall class distribution
class_counts = meta_df["class_name"].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
class_counts.plot(kind="barh", ax=axes[0], color=colors)
axes[0].set_title("Overall Class Distribution (frame counts)", fontsize=14)
axes[0].set_xlabel("Number of annotated frames")

# Per split
class_split = meta_df.groupby(["train_test", "class_name"]).size().unstack(fill_value=0)
class_split.T.plot(kind="bar", ax=axes[1], color=["#4CAF50", "#FF5722"])
axes[1].set_title("Class Distribution: Train vs Test", fontsize=14)
axes[1].set_ylabel("Frame count")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="Split")

plt.tight_layout()
plt.show()


# --- 5. Sony vs TAU Distribution ---
print("\n" + "=" * 60)
print("📊 Sony vs TAU source comparison")
print("=" * 60)

fig, ax = plt.subplots(figsize=(14, 6))
source_class = (
    meta_df.groupby(["source_type", "class_name"]).size().unstack(fill_value=0)
)
source_class.T.plot(kind="bar", ax=ax, color=["#2196F3", "#FF9800"])
ax.set_title("Class Distribution: Sony vs TAU", fontsize=14)
ax.set_ylabel("Frame count")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Source")
plt.tight_layout()
plt.show()


# --- 6. Polyphony Analysis (overlapping events per frame) ---
print("\n" + "=" * 60)
print("📊 Polyphony Analysis")
print("=" * 60)

polyphony = meta_df.groupby(["file_name", "frame"]).size().reset_index(name="n_events")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of polyphony
polyphony["n_events"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color="#9C27B0", edgecolor="white"
)
axes[0].set_title("Polyphony Distribution", fontsize=14)
axes[0].set_xlabel("Number of simultaneous events per frame")
axes[0].set_ylabel("Count")

# Max polyphony per file
max_poly_per_file = polyphony.groupby("file_name")["n_events"].max()
max_poly_per_file.hist(
    bins=range(1, max_poly_per_file.max() + 2),
    ax=axes[1],
    color="#00BCD4",
    edgecolor="white",
)
axes[1].set_title("Max Polyphony per File", fontsize=14)
axes[1].set_xlabel("Max simultaneous events")
axes[1].set_ylabel("Number of files")

plt.tight_layout()
plt.show()

print(f"  Mean polyphony: {polyphony['n_events'].mean():.2f}")
print(f"  Max polyphony:  {polyphony['n_events'].max()}")
print(f"  Median:         {polyphony['n_events'].median():.0f}")


# --- 7. Spatial Distribution (Azimuth & Distance) ---
print("\n" + "=" * 60)
print("📊 Spatial Distribution")
print("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Azimuth histogram
axes[0].hist(meta_df["azimuth"], bins=72, color="#E91E63", edgecolor="white", alpha=0.8)
axes[0].set_title("Azimuth Distribution", fontsize=14)
axes[0].set_xlabel("Azimuth (degrees)")
axes[0].set_ylabel("Count")
axes[0].axvline(0, color="black", linestyle="--", alpha=0.5)

# Distance histogram
if "distance" in meta_df.columns:
    axes[1].hist(
        meta_df["distance"].dropna(),
        bins=50,
        color="#3F51B5",
        edgecolor="white",
        alpha=0.8,
    )
    axes[1].set_title("Distance Distribution", fontsize=14)
    axes[1].set_xlabel("Distance")
    axes[1].set_ylabel("Count")

# Azimuth polar plot
ax_polar = fig.add_subplot(1, 3, 3, projection="polar")
azimuth_rad = np.deg2rad(meta_df["azimuth"])
ax_polar.hist(azimuth_rad, bins=72, color="#FF5722", alpha=0.7)
ax_polar.set_title("Azimuth (Polar View)", fontsize=14, pad=20)
ax_polar.set_theta_zero_location("N")  # 0° at top (front)
ax_polar.set_theta_direction(-1)  # clockwise

plt.tight_layout()
plt.show()

print(f"  Azimuth range:  [{meta_df['azimuth'].min()}°, {meta_df['azimuth'].max()}°]")
if "distance" in meta_df.columns:
    print(
        f"  Distance range: [{meta_df['distance'].min()}, {meta_df['distance'].max()}]"
    )


# --- 8. Onscreen vs Offscreen ---
print("\n" + "=" * 60)
print("📊 Onscreen vs Offscreen")
print("=" * 60)

if "onscreen" in meta_df.columns:
    onscreen_counts = meta_df["onscreen"].value_counts()
    labels_map = {1: "Onscreen", 0: "Offscreen"}

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Pie chart
    axes[0].pie(
        onscreen_counts.values,
        labels=[labels_map.get(k, k) for k in onscreen_counts.index],
        autopct="%1.1f%%",
        colors=["#4CAF50", "#FF5722"],
        startangle=90,
    )
    axes[0].set_title("Onscreen vs Offscreen", fontsize=14)

    # Per class
    onscreen_class = (
        meta_df.groupby(["class_name", "onscreen"]).size().unstack(fill_value=0)
    )
    onscreen_class.columns = [labels_map.get(c, c) for c in onscreen_class.columns]
    onscreen_class.plot(
        kind="barh", stacked=True, ax=axes[1], color=["#FF5722", "#4CAF50"]
    )
    axes[1].set_title("Onscreen/Offscreen per Class", fontsize=14)
    axes[1].set_xlabel("Frame count")

    plt.tight_layout()
    plt.show()


# --- 9. Audio File Analysis ---
print("\n" + "=" * 60)
print("🔊 Audio File Analysis")
print("=" * 60)

# Sample a few audio files for analysis
sample_split = "dev-train-sony"
sample_audio_files = sorted((AUDIO_DIR / sample_split).glob("*.wav"))[:50]

durations = []
sample_rates = []

for af in tqdm(sample_audio_files, desc="Analyzing audio"):
    info = torchaudio.info(str(af))
    durations.append(info.num_frames / info.sample_rate)
    sample_rates.append(info.sample_rate)

print(f"\n  Sample rate: {sample_rates[0]} Hz")
print(f"  Duration range: [{min(durations):.1f}s, {max(durations):.1f}s]")
print(f"  Mean duration:  {np.mean(durations):.1f}s")
print(f"  Channels: {torchaudio.info(str(sample_audio_files[0])).num_channels}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(durations, bins=30, color="#009688", edgecolor="white")
ax.set_title("Audio Duration Distribution (sample of 50 files)", fontsize=14)
ax.set_xlabel("Duration (seconds)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


# --- 10. Visualize a Single File: Audio + Annotations ---
print("\n" + "=" * 60)
print("🎵 Single File Visualization")
print("=" * 60)

# Pick a file with multiple events
sample_file = (
    meta_df.groupby("file_name")
    .apply(lambda x: x["class"].nunique())
    .sort_values(ascending=False)
    .index[0]
)

sample_meta = meta_df[meta_df["file_name"] == sample_file].copy()
sample_audio_path = AUDIO_DIR / sample_meta["split"].iloc[0] / f"{sample_file}.wav"

print(f"  File: {sample_file}")
print(f"  Classes: {sample_meta['class_name'].unique().tolist()}")
print(f"  Frames: {sample_meta['frame'].min()} - {sample_meta['frame'].max()}")

# Load audio
waveform, sr = torchaudio.load(str(sample_audio_path))
time_axis = np.arange(waveform.shape[1]) / sr

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Waveform
axes[0].plot(time_axis, waveform[0].numpy(), color="#2196F3", linewidth=0.3)
axes[0].set_title(f"Waveform — {sample_file}", fontsize=14)
axes[0].set_ylabel("Amplitude")

# Spectrogram
mel_spec = T.MelSpectrogram(sample_rate=sr, n_fft=2048, hop_length=512, n_mels=128)
spec = mel_spec(waveform[0:1])
spec_db = T.AmplitudeToDB()(spec).squeeze().numpy()
axes[1].imshow(
    spec_db,
    aspect="auto",
    origin="lower",
    extent=[0, waveform.shape[1] / sr, 0, sr / 2],
    cmap="magma",
)
axes[1].set_title("Mel Spectrogram", fontsize=14)
axes[1].set_ylabel("Frequency (Hz)")

# Event timeline
frame_duration = 0.1  # 100ms per frame
unique_classes = sample_meta["class_name"].unique()
class_colors = plt.cm.Set2(np.linspace(0, 1, len(unique_classes)))
class_color_map = dict(zip(unique_classes, class_colors))

for _, row in sample_meta.iterrows():
    t = row["frame"] * frame_duration
    y = list(unique_classes).index(row["class_name"])
    axes[2].scatter(t, y, c=[class_color_map[row["class_name"]]], s=3, marker="s")

axes[2].set_yticks(range(len(unique_classes)))
axes[2].set_yticklabels(unique_classes)
axes[2].set_title("Sound Event Timeline", fontsize=14)
axes[2].set_xlabel("Time (seconds)")

plt.tight_layout()
plt.show()

# Play audio
Audio(waveform[0].numpy(), rate=sr)


# --- 11. Summary Statistics ---
print("\n" + "=" * 60)
print("📋 Summary")
print("=" * 60)

summary = {
    "Total files": meta_df["file_name"].nunique(),
    "Total annotation frames": len(meta_df),
    "Sound event classes": meta_df["class"].nunique(),
    "Mean polyphony": f"{polyphony['n_events'].mean():.2f}",
    "Max polyphony": polyphony["n_events"].max(),
    "Azimuth range": f"[{meta_df['azimuth'].min()}°, {meta_df['azimuth'].max()}°]",
    "Train files (sony)": split_info["dev-train-sony"]["metadata_files"],
    "Train files (tau)": split_info["dev-train-tau"]["metadata_files"],
    "Test files (sony)": split_info["dev-test-sony"]["metadata_files"],
    "Test files (tau)": split_info["dev-test-tau"]["metadata_files"],
}

if "onscreen" in meta_df.columns:
    on = (meta_df["onscreen"] == 1).sum()
    off = (meta_df["onscreen"] == 0).sum()
    summary["Onscreen %"] = f"{on / (on + off) * 100:.1f}%"

for k, v in summary.items():
    print(f"  {k:25s}: {v}")

In [ ]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio
from huggingface_hub import login as hf_login
from tqdm.auto import tqdm

# --- Config ---
DATA_DIR = Path.home() / "workspace" / "dataset" / "STARSS22-23"
METADATA_DIR = DATA_DIR / "metadata_dev"
AUDIO_DIR = DATA_DIR / "stereo_dev"

TARGET_SIZE_GB = 3.0

CLASS_MAP = {
    0: "Female speech",
    1: "Male speech",
    2: "Clapping",
    3: "Telephone",
    4: "Laughter",
    5: "Domestic sounds",
    6: "Walk / Footsteps",
    7: "Door (open/close)",
    8: "Music",
    9: "Musical instrument",
    10: "Water tap",
    11: "Bell",
    12: "Knock",
}


# --- 1. Calculate total size & sample fraction ---
def get_split_files(split):
    """Get matched (audio, metadata) file pairs for a split."""
    meta_files = {f.stem: f for f in sorted((METADATA_DIR / split).glob("*.csv"))}
    audio_files = {f.stem: f for f in sorted((AUDIO_DIR / split).glob("*.wav"))}

    # Only keep files that have both audio + metadata
    common = sorted(set(meta_files.keys()) & set(audio_files.keys()))
    return [(audio_files[k], meta_files[k]) for k in common]


def get_total_size(file_pairs):
    """Get total size in GB."""
    return sum(af.stat().st_size for af, _ in file_pairs) / (1024**3)


# Combine train splits and test splits
train_pairs = get_split_files("dev-train-sony") + get_split_files("dev-train-tau")
test_pairs = get_split_files("dev-test-sony") + get_split_files("dev-test-tau")

train_size = get_total_size(train_pairs)
test_size = get_total_size(test_pairs)

print(f"Train: {len(train_pairs)} files, {train_size:.2f} GB")
print(f"Test:  {len(test_pairs)} files, {test_size:.2f} GB")

# Sample to ~3GB each

random.seed(42)


def sample_to_size(file_pairs, target_gb):
    random.shuffle(file_pairs)
    sampled = []
    current_size = 0
    target_bytes = target_gb * (1024**3)

    for af, mf in file_pairs:
        file_size = af.stat().st_size
        if current_size + file_size > target_bytes:
            break
        sampled.append((af, mf))
        current_size += file_size

    print(f"  Sampled {len(sampled)} files ({current_size / (1024**3):.2f} GB)")
    return sampled


print("\nSampling train...")
train_sampled = sample_to_size(train_pairs, TARGET_SIZE_GB)
print("Sampling test...")
test_sampled = sample_to_size(test_pairs, TARGET_SIZE_GB)


# --- 2. Build records ---
def build_records(file_pairs):
    records = []
    for audio_path, meta_path in tqdm(file_pairs, desc="Building records"):
        # Load metadata
        meta_df = pd.read_csv(meta_path)

        # Convert frame-level annotations to a compact string
        # Format: "frame,class,source,azimuth,distance,onscreen;..."
        annotations = meta_df.to_csv(index=False)

        # Extract info from filename
        fname = audio_path.stem
        source_type = "sony" if "sony" in str(audio_path.parent.name) else "tau"

        records.append(
            {
                "file_name": fname,
                "audio": str(audio_path),
                "source_type": source_type,
                "annotations": annotations,  # full CSV as string
                "n_frames": int(meta_df["frame"].max()) + 1 if len(meta_df) > 0 else 0,
                "n_events": int(meta_df["class"].nunique()) if len(meta_df) > 0 else 0,
                "classes": ",".join(
                    sorted(meta_df["class"].map(CLASS_MAP).unique().tolist())
                )
                if len(meta_df) > 0
                else "",
            }
        )
    return records


print("\nBuilding train records...")
train_records = build_records(train_sampled)
print("Building test records...")
test_records = build_records(test_sampled)


# --- 3. Create HF Datasets ---
train_ds = Dataset.from_list(train_records)
test_ds = Dataset.from_list(test_records)

# Cast audio column
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=24000))
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=24000))

ds_dict = DatasetDict(
    {
        "train": train_ds,
        "test": test_ds,
    }
)

print(ds_dict)


# --- 5. To load it later ---
"""
from datasets import load_dataset
import pandas as pd
from io import StringIO

ds = load_dataset("your-username/starss22-23-stereo-sample")
sample = ds["train"][0]

# Access audio
audio = sample["audio"]  # {'array': np.array, 'sampling_rate': 24000}

# Parse annotations back to DataFrame
annotations_df = pd.read_csv(StringIO(sample["annotations"]))
print(annotations_df.head())
#    frame  class  source  azimuth  distance  onscreen
# 0      0      8       0       24       215         1
# 1      1      8       0       24       215         1
"""

In [ ]:
# --- 4. Push to Hub ---
REPO_ID = "8Opt/starss22-23-stereo-sample"  # <-- change this

ds_dict.push_to_hub(
    REPO_ID,
    private=False,
    max_shard_size="2GB",  # split into manageable chunks
)

print(f"✅ Done! Pushed to https://huggingface.co/datasets/{REPO_ID}")